> **Note (2026):** AWS discontinued Amazon Lookout for Vision on **October 31, 2025**, so the console and `lookoutvision` API no longer work.
> The manifest files built here use the standard **SageMaker Ground Truth image-classification format**, so they are still valid. The optional section at the end trains on them with the SageMaker built-in Image Classification algorithm, which is AWS's recommended replacement path.
>
> This notebook never used the SageMaker Python SDK, so nothing here breaks under SDK v3. All AWS calls go through boto3.



## Amazon Lookout for Vision Lab

To help you learn about creating a model, Amazon Lookout for Vision provides example images of circuit boards (circuit_board) that you can use. These images are taken from https://docs.aws.amazon.com/lookout-for-vision/latest/developer-guide/su-prepare-example-images.html.

### Environmental variables

In a very first step we want to define the two global variables needed for this notebook:

- bucket: the S3 bucket that you will create and then use as your source for Amazon Lookout for Vision
    - Note: Please read the comments carefully. Depending on your region you need to uncomment the correct command
- project: the project name you want to use in Amazon Lookout for Vision

In [4]:
import os
import boto3

bucket = "sagemaker-us-east-1-155361516759"
project = "circuitproject"
region = boto3.session.Session().region_name
os.environ["BUCKET"] = bucket
os.environ["REGION"] = region

# client = boto3.client('lookoutvision')   # service discontinued Oct 31, 2025
s3_client = boto3.client('s3', region_name=region)
sm_client = boto3.client('sagemaker', region_name=region)
print(sm_client)

You can check your region here with:

In [5]:
# Check your region:
print(boto3.session.Session().region_name)

us-east-1


Depending on your region follow the instructions of the next cell:

In [6]:
# Create the bucket if it doesn't exist yet.
# us-east-1 must NOT pass a LocationConstraint; every other region must.
from botocore.exceptions import ClientError

try:
    s3_client.head_bucket(Bucket=bucket)
    print(f"Bucket {bucket} already exists")
except ClientError:
    if region == "us-east-1":
        s3_client.create_bucket(Bucket=bucket)
    else:
        s3_client.create_bucket(Bucket=bucket,
                                CreateBucketConfiguration={"LocationConstraint": region})
    print(f"Created bucket {bucket} in {region}")

Bucket sagemaker-us-east-1-155361516759 already exists


## Image Preparation and EDA

In Amazon Lookout for Vision - see also
- https://aws.amazon.com/lookout-for-vision/ and
- https://aws.amazon.com/blogs/aws/amazon-lookout-for-vision-new-machine-learning-service-that-simplifies-defect-detection-for-manufacturing/
if you already have pre-labeled images available, as it is the case in this example, you can already establish a folder structure that lets you define training and validation. Further, images are labeled for Amazon Lookout via the corresponding folder (normal=good, anomaly=bad).

We will import the sample images provided by AWS Lookout of Vision. If you're importing your own images, you will prepare them at this stage.

### Generate the *manifest* files

You might be familiar with the manifest files if you ever used Amazon SageMaker Ground Truth. If you are not don't worry about that section too much.

If you are still interested in what's happening, you can continue reading:

Each dataset training/ as well as validation/ needs a manifest file. This file is used by Amazon Lookout for Vision to determine where to look for the images. The manifest follows a fixed structure. Most importantly are the keys (it's JSON formatted) *source-ref* this is the location for each file, *auto-label* the value for each label (0=bad, 1=good), *folder* which indicates whether Amazon Lookout is using training or validation and *creation-date* as this let's you know when an image was put in place. All other fields are pre-set for you.

Each manifest file itself contains N JSON objects, where N is the number of images that are used in this dataset.

In [8]:
!unzip circuitboard.zip

Archive:  circuitboard.zip
   creating: circuitboard/
   creating: circuitboard/extra_images/
  inflating: circuitboard/extra_images/extra_images-anomaly_1.jpg  
  inflating: circuitboard/extra_images/extra_images-anomaly_10.jpg  
  inflating: circuitboard/extra_images/extra_images-anomaly_2.jpg  
  inflating: circuitboard/extra_images/extra_images-anomaly_3.jpg  
  inflating: circuitboard/extra_images/extra_images-anomaly_4.jpg  
  inflating: circuitboard/extra_images/extra_images-anomaly_5.jpg  
  inflating: circuitboard/extra_images/extra_images-anomaly_6.jpg  
  inflating: circuitboard/extra_images/extra_images-anomaly_7.jpg  
  inflating: circuitboard/extra_images/extra_images-anomaly_8.jpg  
  inflating: circuitboard/extra_images/extra_images-anomaly_9.jpg  
  inflating: circuitboard/extra_images/extra_images-normal_1.jpg  
  inflating: circuitboard/extra_images/extra_images-normal_10.jpg  
  inflating: circuitboard/extra_images/extra_images-normal_2.jpg  
  inflating: circuitboa

In [9]:
# Datetime for datetime generation and json to dump the JSON object
# to the corresponding files:
from datetime import datetime
import json

# Current date and time in manifest file format:
dttm = datetime.now().strftime("%Y-%m-%dT%H:%M:%S.%f")

IMAGE_EXT = ('.jpg', '.jpeg', '.png')
MAX_PER_CLASS = 20      # set to None to use the entire dataset

# The two datasets used: train and test
datasets = ["train", "test"]
counts = {}

for ds in datasets:
    # Only keep real class folders (skips .ipynb_checkpoints, .DS_Store, ...)
    folders = sorted(d for d in os.listdir(f"./circuitboard/{ds}")
                     if os.path.isdir(f"./circuitboard/{ds}/{d}") and not d.startswith('.'))
    n = 0
    with open(f"{ds}.manifest", "w") as f:
        for folder in folders:
            # Label convention: 1 = good (normal), 0 = bad (anomaly)
            label = 0 if folder == "anomaly" else 1
            files = sorted(x for x in os.listdir(f"./circuitboard/{ds}/{folder}")
                           if x.lower().endswith(IMAGE_EXT))
            if MAX_PER_CLASS:
                files = files[:MAX_PER_CLASS]
            for file in files:
                manifest = {
                    "source-ref": f"s3://{bucket}/{project}/{ds}/{folder}/{file}",
                    "auto-label": label,
                    "auto-label-metadata": {
                        "confidence": 1,
                        "job-name": "labeling-job/auto-label",
                        "class-name": folder,
                        "human-annotated": "yes",
                        "creation-date": dttm,
                        "type": "groundtruth/image-classification"
                    }
                }
                f.write(json.dumps(manifest) + "\n")
                n += 1
            print(f"{ds}/{folder}: {len(files)} images, label={label}")
    counts[ds] = n

print(counts)

train/anomaly: 20 images, label=0
train/normal: 20 images, label=1
test/anomaly: 20 images, label=0
test/normal: 20 images, label=1
{'train': 40, 'test': 40}


### Upload manifest files and images to S3

Now it's time to upload all the images and the manifest files:

In [10]:
# Upload manifest files to S3 bucket:
for ds in datasets:
    key = f"{project}/{ds}.manifest"
    s3_client.upload_file(f"{ds}.manifest", bucket, key)
    print(f"s3://{bucket}/{key}")

s3://sagemaker-us-east-1-155361516759/circuitproject/train.manifest
s3://sagemaker-us-east-1-155361516759/circuitproject/test.manifest


In [11]:
# Upload images to S3 bucket (same layout as the old `aws s3 cp --recursive` calls):
for ds in datasets:
    for folder in ["normal", "anomaly"]:
        local_dir = f"circuitboard/{ds}/{folder}"
        files = [x for x in os.listdir(local_dir) if x.lower().endswith(IMAGE_EXT)]
        for file in files:
            s3_client.upload_file(os.path.join(local_dir, file), bucket, f"{project}/{ds}/{folder}/{file}")
        print(f"{local_dir}: uploaded {len(files)} files -> s3://{bucket}/{project}/{ds}/{folder}/")

# Equivalent CLI version, if you prefer:
# !aws s3 cp circuitboard/train s3://{bucket}/{project}/train --recursive
# !aws s3 cp circuitboard/test  s3://{bucket}/{project}/test  --recursive

circuitboard/train/normal: uploaded 20 files -> s3://sagemaker-us-east-1-155361516759/circuitproject/train/normal/
circuitboard/train/anomaly: uploaded 20 files -> s3://sagemaker-us-east-1-155361516759/circuitproject/train/anomaly/
circuitboard/test/normal: uploaded 20 files -> s3://sagemaker-us-east-1-155361516759/circuitproject/test/normal/
circuitboard/test/anomaly: uploaded 20 files -> s3://sagemaker-us-east-1-155361516759/circuitproject/test/anomaly/


## (Optional) Train on the manifests with SageMaker

Since Lookout for Vision is gone, this trains a binary classifier with the SageMaker built-in **Image Classification** algorithm. The algorithm reads the Ground Truth manifests directly as an *augmented manifest* in Pipe mode, so the images don't need to be converted.

⚠️ This uses a **GPU instance** (billed). Check your quota for `ml.g4dn.xlarge` or switch the instance type to one you have access to.

In [12]:
import time

try:
    from sagemaker.core.helper.session_helper import get_execution_role
except ImportError:
    from sagemaker import get_execution_role
role = get_execution_role()

try:
    from sagemaker.core import image_uris
except ImportError:
    from sagemaker import image_uris
container = image_uris.retrieve('image-classification', region)
print(container)

def manifest_channel(name, ds):
    return {
        "ChannelName": name,
        "DataSource": {"S3DataSource": {
            "S3DataType": "AugmentedManifestFile",
            "S3Uri": f"s3://{bucket}/{project}/{ds}.manifest",
            "S3DataDistributionType": "FullyReplicated",
            "AttributeNames": ["source-ref", "auto-label"],
        }},
        "ContentType": "application/x-recordio",
        "RecordWrapperType": "RecordIO",
        "CompressionType": "None",
    }

job_name = 'circuitboard-ic-' + time.strftime("%Y-%m-%d-%H-%M-%S", time.gmtime())

training_params = {
    "TrainingJobName": job_name,
    "RoleArn": role,
    "AlgorithmSpecification": {"TrainingImage": container, "TrainingInputMode": "Pipe"},
    "ResourceConfig": {"InstanceCount": 1, "InstanceType": "ml.g4dn.xlarge", "VolumeSizeInGB": 30},
    "InputDataConfig": [manifest_channel("train", "train"), manifest_channel("validation", "test")],
    "OutputDataConfig": {"S3OutputPath": f"s3://{bucket}/{project}/output/"},
    "HyperParameters": {
        "num_classes": "2",
        "num_training_samples": str(counts["train"]),
        "use_pretrained_model": "1",       # transfer learning, important with few images
        "image_shape": "3,224,224",
        "epochs": "10",
        "learning_rate": "0.001",
        "mini_batch_size": str(min(16, counts["train"])),
        "augmentation_type": "crop_color_transform",
    },
    "StoppingCondition": {"MaxRuntimeInSeconds": 60 * 60},
}
print("Job name is:", job_name)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


[10/28/26 19:41:42] INFO     Same images used for training and inference. Defaulting to image     ]8;id=10337528;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/image_uris.py\image_uris.py]8;;\:]8;id=10337529;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/image_uris.py#408\408]8;;\
                             scope: inference.                                                                     

                    INFO     Ignoring unnecessary instance type: None.                            ]8;id=10337535;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/image_uris.py\image_uris.py]8;;\:]8;id=10337536;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/image_uris.py#545\545]8;;\

811284229777.dkr.ecr.us-east-1.amazonaws.com/image-classification:1
Job name is: circuitboard-ic-2026-10-28-19-41-42


In [15]:
%%time
sm_client.create_training_job(**training_params)
sm_client.get_waiter('training_job_completed_or_stopped').wait(TrainingJobName=job_name)

desc = sm_client.describe_training_job(TrainingJobName=job_name)
print("Training job ended with status:", desc['TrainingJobStatus'])
if desc['TrainingJobStatus'] == 'Failed':
    raise Exception(desc['FailureReason'])
for m in desc.get('FinalMetricDataList', []):
    print('   ', m['MetricName'], round(m['Value'], 4))

Training job ended with status: Completed
    validation:accuracy:epoch 0.625
    train:accuracy:epoch 0.4688
    validation:accuracy 0.625
    train:accuracy 0.4688
CPU times: user 7.84 ms, sys: 7.44 ms, total: 15.3 ms
Wall time: 62.1 ms
